# Initially, we need to get everything installed, imported, downloaded and setup. We will do this by analyzing various documents from Project Gutenburg to use as input texts. Later, this will be the initial source of any paragraph/sentence/word embeddings input we may make.

In [95]:
!pip install nltk numpy scikit-learn

In [111]:
# Imports :D

import nltk
import re
import string
import unicodedata
import urllib.request
import numpy as np
import random
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mason\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [97]:
# These are the books we are analyzing and using as input data.

books = {
    "dracula": "https://www.gutenberg.org/cache/epub/345/pg345.txt",
    "frankenstein": "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "war_of_the_worlds": "https://www.gutenberg.org/cache/epub/36/pg36.txt",
    "sherlock": "https://www.gutenberg.org/cache/epub/1661/pg1661.txt",
}

raw_books = {}

for title, url in books.items():
    response = urllib.request.urlopen(url)
    text = response.read().decode("utf-8")
    raw_books[title] = text
    print(f"Downloaded {title}, {len(text)} characters long")


Downloaded dracula, 881021 characters long
Downloaded frankenstein, 446544 characters long
Downloaded war_of_the_worlds, 363420 characters long
Downloaded sherlock, 593871 characters long


Now, we have imported/downloaded the proper things. Now we can begin normalizing, cleaning, general preprocessing, and data splitting to prepare out inputs for embedding.

In [98]:
# This is to normalize the text by removing weird junk in the data, lowercasing, removing whitespace, etc.
# Do for all downloaded books.

def normalize_text(text):
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")

    start_match = re.search(r"\*\*\* start of (.*?) \*\*\*", text, flags=re.IGNORECASE)
    end_match = re.search(r"\*\*\* end of (.*?) \*\*\*", text, flags=re.IGNORECASE)

    if start_match and end_match:
        text = text[start_match.end():end_match.start()]

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip().lower()

    return text

clean_books = {title: normalize_text(text) for title, text in raw_books.items()}


In [99]:
# Split into paragraphs and tokenize sentences/words

def split_into_paragraphs(text):
    paragraphs = re.split(r'\n\s*\n', text)                         # Criteria for splitting
    return [p.strip() for p in paragraphs if len(p.strip()) > 50]

def tokenize_paragraph(paragraph):
    sentences = sent_tokenize(paragraph)        # Function that splits a paragraph into sentences from nltk :) same w words below for worrd_tokenize
    tokenized_sentences = []

    for s in sentences:
        tokens = word_tokenize(s)
        tokens = [t for t in tokens if t not in string.punctuation]
        tokenized_sentences.append(tokens)

    return tokenized_sentences

books_paragraphs = {}

for title, text in clean_books.items():
    paragraphs = split_into_paragraphs(text)
    tokenized_paragraphs = [tokenize_paragraph(p) for p in paragraphs]      # The tokenization process
    books_paragraphs[title] = tokenized_paragraphs

# Sanity check
for title, paragraphs in books_paragraphs.items():
    print(f"{title} paragraphs:", len(paragraphs))
    for p in paragraphs[:5]:
        print("-", sum(len(w) for s in p for w in s), "chars in this paragraph (approx)")


dracula paragraphs: 1612
- 75 chars in this paragraph (approx)
- 58 chars in this paragraph (approx)
- 794 chars in this paragraph (approx)
- 374 chars in this paragraph (approx)
- 496 chars in this paragraph (approx)
frankenstein paragraphs: 721
- 235 chars in this paragraph (approx)
- 238 chars in this paragraph (approx)
- 1774 chars in this paragraph (approx)
- 780 chars in this paragraph (approx)
- 426 chars in this paragraph (approx)
war_of_the_worlds paragraphs: 775
- 138 chars in this paragraph (approx)
- 342 chars in this paragraph (approx)
- 179 chars in this paragraph (approx)
- 1083 chars in this paragraph (approx)
- 493 chars in this paragraph (approx)
sherlock paragraphs: 1738
- 323 chars in this paragraph (approx)
- 918 chars in this paragraph (approx)
- 1054 chars in this paragraph (approx)
- 765 chars in this paragraph (approx)
- 262 chars in this paragraph (approx)


After splitting and normalizing and whatnot, we will flatten the tokens into a vocab for further analysis.

In [100]:
# Flatten tokens to create the vocabulary

all_tokens = []

for paragraphs in books_paragraphs.values():
    for para in paragraphs:
        for sent in para:
            all_tokens.extend(sent)

# Map each word to an index
vocab = {w: i for i, w in enumerate(sorted(set(all_tokens)))}
print("Vocabulary size:", len(vocab))


Vocabulary size: 18586


We then create the embeddings

In [101]:
# Creates the actual embeddings :D !

# Word embeddings

def word_embedding(word, vocab):
    vec = np.zeros(len(vocab))

    if word in vocab:
        vec[vocab[word]] = 1.0

    return vec

# Returns the mean of the word embeddings

def sentence_embedding(tokens, vocab):
    if not tokens:
        return np.zeros(len(vocab))
    
    vecs = [word_embedding(w, vocab) for w in tokens]

    return np.mean(vecs, axis=0)

# Returns the mean of the sentence embeddings

def paragraph_embedding(paragraph_sentences, vocab):
    if not paragraph_sentences:
        return np.zeros(len(vocab))
    
    sent_vecs = [sentence_embedding(s, vocab) for s in paragraph_sentences]

    return np.mean(sent_vecs, axis=0)


Sanity check

In [102]:
# First 3 paragraphs of Dracula
example_paragraphs = books_paragraphs["dracula"][:3]
for i, para in enumerate(example_paragraphs):
    p_emb = paragraph_embedding(para, vocab)
    print(f"Paragraph {i} embedding shape: {p_emb.shape}")


Paragraph 0 embedding shape: (18586,)
Paragraph 1 embedding shape: (18586,)
Paragraph 2 embedding shape: (18586,)


Now two examples showing an example of how sentences close to each other are significantly higher (in terms of cosine similarity of mean embeddings) as opposed to sentences from different books. This is an expected observation, as different books probably have very different words in them. 

In [ ]:
p1 = paragraph_embedding(books_paragraphs["dracula"][7], vocab)
p2 = paragraph_embedding(books_paragraphs["dracula"][9], vocab)
p3 = paragraph_embedding(books_paragraphs["frankenstein"][152], vocab)

print("Cosine similarity between (Dracula e1, Dracula e2):", cosine_similarity([p1], [p2])[0][0])
print("Cosine similarity between (Dracula e1, Frankenstein e1):", cosine_similarity([p1], [p3])[0][0])


Similarity(Dracula e1, Dracula e2): 0.5019144328967918
Similarity(Dracula e1, Frankenstein e1): 0.14618121824667787


This example shows that even the same book can have very different embeddings though, and farther apart can cause this, as well as potentially having this first embedding being an introductory paragraph.

In [ ]:
p1 = paragraph_embedding(books_paragraphs["dracula"][0], vocab)
p2 = paragraph_embedding(books_paragraphs["dracula"][9], vocab)
p3 = paragraph_embedding(books_paragraphs["frankenstein"][152], vocab)

print("Cosine similarity between (Dracula e1, Dracula e2):", cosine_similarity([p1], [p2])[0][0])
print("Cosine similarity between (Dracula e1, Frankenstein e1):", cosine_similarity([p1], [p3])[0][0])


Similarity(Dracula e1, Dracula e2): 0.33812541799829393
Similarity(Dracula e1, Frankenstein e1): 0.18168705651380831


This shows very interesting results I think. Firstly, it shows that the first introductory paragraphs actually have no similarity - an odd but interesting observation.
Furthermore, the similarities between the intro and the two second paragraphs is approximately the same. This too, is interesting. It shows that the introduction paragraph is incredibly unique, not similar necessarily to the book it is apart of, and not even similar to other introductions. Perhaps this needs morer literary study to understand the meaning of but interesting nonetheless.

In [ ]:
p1 = paragraph_embedding(books_paragraphs["dracula"][0], vocab)
p2 = paragraph_embedding(books_paragraphs["dracula"][1], vocab)
p3 = paragraph_embedding(books_paragraphs["frankenstein"][0], vocab)
p4 = paragraph_embedding(books_paragraphs["frankenstein"][1], vocab)

print("Cosine similarity between (Dracula e1, Dracula e2):", cosine_similarity([p1], [p2])[0][0])
print("Cosine similarity between (Dracula e1, Frankenstein e1):", cosine_similarity([p1], [p3])[0][0])
print("Cosine similarity between (Dracula e1, Frankenstein e1):", cosine_similarity([p1], [p4])[0][0])


Similarity(Dracula e1, Dracula e2): 0.30429030972509236
Similarity(Dracula e1, Frankenstein e1): 0.0
Similarity(Dracula e1, Frankenstein e1): 0.2981037701780714


Finally some auto tests to sort through, chosen mostly randomly between two books. Just more datapoints for analysis. More analysis after I run it haha ...

In [123]:
iterations = 40

for i in range(iterations):
     book_name_1 = random.choice(list(books.keys()))
     book_name_2 = random.choice(list(books.keys()))

     paragraph_1 = random.randint(0, len(books_paragraphs[book_name_1]) - 1)
     paragraph_2 = random.randint(0, len(books_paragraphs[book_name_2]) - 1)

     one = paragraph_embedding(books_paragraphs[book_name_1][paragraph_1], vocab)
     two = paragraph_embedding(books_paragraphs[book_name_2][paragraph_2], vocab)

     print(f"The cosine similarity between {book_name_1} paragraph {paragraph_1} and {book_name_2} paragraph {paragraph_2} is: {cosine_similarity([one], [two])[0][0]}\033[1m")


The cosine similarity between frankenstein paragraph 386 and dracula paragraph 523 is: 0.0
The cosine similarity between sherlock paragraph 1571 and frankenstein paragraph 545 is: 0.24065239307233027
The cosine similarity between dracula paragraph 86 and war_of_the_worlds paragraph 341 is: 0.39339613004933366
The cosine similarity between sherlock paragraph 298 and dracula paragraph 173 is: 0.17957643274509927
The cosine similarity between dracula paragraph 738 and sherlock paragraph 660 is: 0.07690835077671787
The cosine similarity between dracula paragraph 165 and war_of_the_worlds paragraph 354 is: 0.05693810744294761
The cosine similarity between war_of_the_worlds paragraph 729 and war_of_the_worlds paragraph 200 is: 0.2539486212128981
The cosine similarity between dracula paragraph 1384 and sherlock paragraph 815 is: 0.0470772803417788
The cosine similarity between war_of_the_worlds paragraph 290 and dracula paragraph 117 is: 0.5746962323019158
The cosine similarity between war_of

# Overall, these are interesting results. Of course, these are completely random. However, they do depict some relationship between the same book having higher similarity as it is with some of the dracula to dracula examples. Further, there are examples where there is high similarity between different books which is interesting! Of course, some of these being both monster books, and all of these large books, that is probable and not extremely unsurprising. There seems to be a lot of randomness. The highest similarity I see are between frankenstein paragraphs a few hundred paragraphs apart with nearly .7 as a similarity score! That is pretty impressive. 

# Additionally, it seems to me that there are lots of notable observations from this and there is a lot of randomness, some of the expected things are described in the premeditated trials above though. As for this part of our experiments I believe this does an excellent job of showing not only the plausability, but also some possible potential uses of this paragraph embedding analysis.

# Moreover, independent research showed some strategies like Doc2Vec and some things like neural nets being utilized to analyze entire paragraphs in an embedding-esque system. This is interesting and seems to be an avenue of the future for sure.

# Moving on, there are further experiments that will later be explained and described: